[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Loading Strategies &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: `counting`, `SPRING_SECTIONS` and `class_list_lines`. Run it first. The tasks do not
depend on one another, and the last cell removes the scratch folder.


In [1]:
import logging
import re
import shutil
from contextlib import contextmanager
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event, func,
                        insert, select)
from sqlalchemy.orm import (DeclarativeBase, Mapped, Session, joinedload, mapped_column, raiseload, relationship,
                            selectinload, sessionmaker)
from sqlalchemy.exc import InvalidRequestError
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


def without_address(error):
    """An error's message with every memory address replaced, since the addresses change on every run."""
    return re.sub(r"0x[0-9a-f]+", "0x...", str(error))


@contextmanager
def counting(engine):
    """Count the statements an engine sends while the block runs, in a dictionary the block can read."""
    counter = {"statements": 0}

    def count(conn, cursor, statement, parameters, context, executemany):
        counter["statements"] += 1

    event.listen(engine, "before_cursor_execute", count)
    try:
        yield counter
    finally:
        event.remove(engine, "before_cursor_execute", count)


SPRING_SECTIONS = select(Section).where(Section.term_id == 4).order_by(Section.id)


def class_list_lines(sections):
    """One line for every section: its course, how many students it has, and the first three of them."""
    lines = []
    for section in sections:
        names = [enrollment.student.name for enrollment in section.enrollments]
        lines.append(f"{section.course.code}  {section.course.title:<27} {len(names)} students: {', '.join(names[:3])}")
    return lines


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


**1.** A loop over History students, lazily and eagerly.


In [2]:
HISTORY = select(Student).where(Student.program == "History").order_by(Student.name)

for label, query in [("lazy", HISTORY), ("selectinload", HISTORY.options(selectinload(Student.enrollments)))]:
    with SessionLocal() as session, counting(engine) as sent:
        counts = [len(student.enrollments) for student in session.scalars(query)]
    print(f"{label:<13} {sent['statements']} statements for {counts}")


lazy          6 statements for [12, 9, 12, 6, 9]
selectinload  2 statements for [12, 9, 12, 6, 9]


Six statements lazily, one for the students and one for each of the five, and two with
`selectinload`.


**2.** A transcript in two statements.


In [3]:
TRANSCRIPT = select(Student).where(Student.id == 3).options(
    selectinload(Student.enrollments)
    .joinedload(Enrollment.section)
    .options(joinedload(Section.course), joinedload(Section.term))
)
with SessionLocal() as session, counting(engine) as sent:
    chloe = session.scalars(TRANSCRIPT).one()
    lines = [(e.section.term.name, e.section.course.code, e.grade) for e in chloe.enrollments]
print(lines)
print("statements:", sent["statements"])


[('Fall 2025', 'CHE-110', 'C+'), ('Fall 2025', 'CSC-201', 'F'), ('Fall 2025', 'PSY-101', 'B+'), ('Spring 2026', 'MAT-120', None), ('Spring 2026', 'ENG-105', None), ('Spring 2026', 'STA-200', None)]
statements: 2


The student in one statement, and her enrollments in a second, with each enrollment's section, and
the section's course and term, joined into that one. `.options()` after a loader chains two
strategies from the same step, the course and the term of every section.


**3.** Two joined relationships in one query.


In [4]:
FALL = select(Section).where(Section.term_id == 3).options(joinedload(Section.course), joinedload(Section.term))
print(" ".join(str(FALL.compile(engine)).split()))


SELECT sections.id, sections.course_id, sections.term_id, sections.capacity, courses_1.id AS id_1, courses_1.code, courses_1.title, courses_1.department, courses_1.credits, terms_1.id AS id_2, terms_1.name, terms_1.starts_on FROM sections LEFT OUTER JOIN courses AS courses_1 ON courses_1.id = sections.course_id LEFT OUTER JOIN terms AS terms_1 ON terms_1.id = sections.term_id WHERE sections.term_id = ?


Two `LEFT OUTER JOIN`s, one for each many to one relationship, under the aliases `courses_1` and
`terms_1`. Every section has exactly one of each, so the rows do not multiply.


**4.** Two steps of `selectinload`.


In [5]:
with SessionLocal() as session, counting(engine) as sent:
    courses = session.scalars(select(Course).options(selectinload(Course.sections).selectinload(Section.term))).all()
    sections = [(section.course_id, section.term.name) for course in courses for section in course.sections]
print(len(courses), "courses,", len(sections), "sections | statements:", sent["statements"])


10 courses, 40 sections | statements: 3


Three statements: the courses, their sections, and the sections' terms, one for every step of the
path.


**5.** Finding the relationships a function reads, with `raiseload`.


In [6]:
attempts = [
    [raiseload("*")],
    [selectinload(Section.enrollments), raiseload("*")],
    [selectinload(Section.enrollments).joinedload(Enrollment.student), raiseload("*")],
    [selectinload(Section.enrollments).joinedload(Enrollment.student), joinedload(Section.course), raiseload("*")],
]
for options in attempts:
    with SessionLocal() as session:
        try:
            class_list_lines(session.scalars(SPRING_SECTIONS.options(*options)))
            print("ran, with every relationship it reads loaded")
        except InvalidRequestError as error:
            print("refused:", error)


refused: 'Section.enrollments' is not available due to lazy='raise'
refused: 'Enrollment.student' is not available due to lazy='raise'
refused: 'Section.course' is not available due to lazy='raise'
ran, with every relationship it reads loaded


Every refusal names the next relationship to load, in the order `class_list_lines` reads them: the
enrollments, their students, and the course. With all three loaded, it ran.


**6.** A loaded collection, read after the session closed.


In [7]:
with SessionLocal() as session:
    sections = session.scalars(SPRING_SECTIONS.options(selectinload(Section.enrollments))).all()

print([(section.id, len(section.enrollments)) for section in sections])


[(31, 8), (32, 8), (33, 7), (34, 7), (35, 8), (36, 7), (37, 7), (38, 8), (39, 8), (40, 7)]


Every collection was loaded while the session was open, so reading it after the close needed no
query.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Loading Strategies](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/13-loading-strategies.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
